# DIAMOND: Sterowanie Trajektorią w Celu Usunięcia Artefaktów w Modelach Flow Matching

![DIAMOND Diagram](img/diamond_diag.png)

Modele generatywne tworzące obrazy (takie jak FLUX czy SDXL) osiągnęły niesamowity poziom fotorealizmu. Niestety, wciąż borykają się z problemami anatomicznymi, strukturalnymi (np. wygenerowanie sześciu palców u dłoni, połamane kończyny) oraz poprawną reprezentacją tekstu na obrazie. 

Dotychczasowe metody naprawy tych błędów były inwazyjne – wymagały kosztownego dotrenowywania modeli (LoRA) albo ciężkich obliczeniowo poprawek już po wygenerowaniu obrazu (post-hoc).

Diamond (*Directed Inference for Artifact Mitigation in Flow Matching Models*) jest techniką kierowania trajektorią, który działa w trybie zero-shot i nie igneruje w model zajmujący się procesem generacji. 

Mechanizm działania polega na prognozowaniu wyglądu obiektu po odszumianiu na każdym kroku generacji, detekcje niechcianych artefaktów i edycję trajektorii w porządanym kierunku.

In [ ]:
!git clone https://github.com/gmum/DIAMOND
%cd DIAMOND

Cloning into 'DIAMOND'...
remote: Enumerating objects: 321, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 321 (delta 16), reused 2 (delta 0), pack-reused 288 (from 1)
Receiving objects: 100% (321/321), 769.29 MiB | 34.47 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/home/bartosz/Desktop/Studia/RepresentationDL/trajectory-guidence/DIAMOND


## Jak działa DIAMOND?

DIAMOND działa w procesie odszumiania na zasadzie ciągłej oceny czystości obrazu (korekcja trajektorii krok po kroku). Proces nie czeka do wygenerowania gotowego obrazu by go naprawić, składa się z:

1. Odtworzenie Odszumionego Obrazu: Z modelu (szczególnie tych opartych o *Rectified Flow* takich jak FLUX, gdzie trajektorie są możliwie jak najprostsze/liniowe) wyciągamy predykcję prędkości wektora (velocity). Model oblicza, jak mógłby wyglądać zdekodowany i całkowicie wolny od szumu obraz docelowy $\hat{x}_{0,t}$ w aktualnym kroku $t$.
2. Detekcja Błędu: Szacunek przekazujemy do ogromnie ważnej części całego procesu - Detektora Artefaktów (Artifact Detector). Sieć ta skanuje obraz i weryfikuje za pomocą stworzonej maski, w którym miejscach pikseli powstanie artefakt (np. dodatkowy palec).
3. Obliczenie Błędu i Gradienty: Dzięki liniowości FLUX (Rectified Flow), proces generowania opiera się na prostych, różniczkowalnych transformacjach. Pozwala nam to  wyliczyć gradient błędu z straty zwróconej przez Detektor i skorygować trajektorię obrazu w czasie rzeczywistym. Sama strata jest z charakteru pixel wide, tzn. tylko obszary odpowiadające za niechciany artefakt zostaną zkorygowane.
4. Odpychanie Trajektorii: W naszym kroku *t* modyfikujemy bazowe pole, zachowując przy tym dynamiczność - kroki na początku są dużo ważniejsze niż pod koniec generacji, co znaczy, że podczas powstawania szkieletu obiektu zmiany trajektorii będą dużo silniejsze. Wartośc naszego kroku zależy więc od czasu i gradientów.


> *Dla przypomnienia z poprzedniej części: Dyfuzja oparta o tzw. Flow-Matching pozwala łączyć startowy próbkowany szum $x_1$ i wygenerowany ostatecznie obraz $x_0$ całkowicie liniowymi drogami. Sprawia to, że możemy wyliczyć predykcję końcową szybciej i prościej.*


![Flow Klatek](img/diamond_bball_sign.png)

### Matematyka użyta w projekcie: Liniowe przesuwanie trajektorii (Rectified Flow)

Istotą Flow Matchingu jest to, że staramy się uczyć ścieżek transportujących dane z szumu $x_1$ do obrazu docelowego $x_0$ prostymi liniami:
$$x_t = (1 - t)x_0 + t x_1$$
Dzięki relacjom matematycznym tej ścieżki i tzw. "prędkości" $v_\theta(x_t, t)$, model przewiduje docelowy, czysty obraz (clean data latent) w każdym dowolnym kroku czasowym podczas tworzenia go na samym "żywym" szumie:

$$\hat{x}_{0,t} = x_t - t \cdot v_\theta(x_t, t)$$

Głównym elementem "naprawy w locie" jest modyfikacja kroku obliczeniowego - całkowania Eulera, którego używa solver modelu dodając do niej wektor kierunkowy przeskalowany odpowiednim suwakiem skali $s_t$:

$$ x_{t-\Delta t} = x_t - \Delta t \cdot \left[ v_\theta(x_t,t) + s_t \cdot \nabla_{x_t}\mathcal{L}_{artifact}(\hat{x}_{0,t}) \right]$$

---

![Gradient normalization impact](img/diamond_norm_grad.png)

---

## Kluczowe Cechy Metodologii DIAMOND

Aby powyższy proces działał stabilnie potrzebne są:

1. **Normalizacja Gradientu (Gradient Normalization):**
Naiwne odjęcie wektora $\nabla_{x_t} \mathcal{L}_a$ powoduje, że norma gradientu bardzo rośnie, jeśli na docelowym obrazku znajduje się wiele artefaktów lub sam artefakt jest duży, dokonamy gwałtownej zmiany na obrazku, skutkującej w błędnym wyniku.
Zabieg optymalizujący sprowadza się do usunięcia "siły" gradientu za pomocą stałego wektora:
$$ \delta_t = \lambda_t \frac{\nabla_{x_t} L_a}{||\nabla_{x_t} L_a||_2 + \epsilon} $$
Oddzielenie siły (określanej od razu przez skalar przypisany do kroku generacji $\lambda_t$) od kierunku gradientu upewnia nas w tym, że proces zawsze pozostanie spójny, niezaleznie od rozmiaru korygowanej cechy, jedyne co wpływa na siłę to $\lambda_t$.

2. **Skalowanie w czasie (Power Schedule, $\lambda_t$):**
Podczas pracy nad każdym krokiem w dyfuzji, początkowo generowany jest zarys kształtów globalnych, a detale (włosy, cienie, ostrości) zachodzą blisko końca tworzenia (**high-frequency details**).
Artefakty takie jak "dziwne ułożenie palców", "dodatkowa liczba kończyn" lub "niewyraźne/zepsute litery" to problemy strukturalne - powstają w już na początku. Zależność parametrów naprawy DIAMOND ustala się za pomocą funkcji wielomianowej, siła maleje potęgowo wraz z każdym kolejnym krokiem:
$$\lambda_t = \lambda_{end} + (\lambda_{start} - \lambda_{end}) \left( 1 - \frac{i}{N - 1} \right)^p$$

## Zadanie – Pojedynczy krok korekcji trajektorii w DIAMOND

DIAMOND potrafi naprawiać generację *w locie* podczas inferencji – używa do tego **Artifact Detector**, który wykrywa defekty w bieżącym obrazie i odpycha trajektorię od nich przez gradienty.

Twoim zadaniem jest uzupełnienie funkcji `step_diamond`, która wykonuje jeden taki krok korekcyjny. Miejsca do uzupełnienia są oznaczone `???`.

**Trzy rzeczy do zrobienia:**
1. Z wektora stanu $x_t$ i przewidzianej prędkości $v_\theta$ odtwórz estymację czystego obrazu $\hat{x}_0$
2. Przepuść ją przez detektor artefaktów i policz loss
3. Zaktualizuj stan $x_t$ o krok $dt$, uwzględniając poprawkę gradientową

In [ ]:
import torch
import torch.nn as nn

class DummyArtifactDetector(nn.Module):
    def forward(self, clean_image_estimate):
        # Im wyższy wynik, tym więcej artefaktów na obrazie
        loss = torch.sum(clean_image_estimate ** 2)
        return loss

class DummyFlowModel:
    def __init__(self):
        self.artifact_detector = DummyArtifactDetector()

    def get_velocity(self, x_t, t):
        # Sieć przepływu przewiduje prędkość v_theta(x_t, t)
        return torch.randn_like(x_t)

    def step_diamond(self, x_t, t, dt, guidance_scale=1.0):
        # Śledzimy gradienty względem x_t
        x_t = x_t.clone().detach().requires_grad_(True)

        # Predykcja prędkości
        v = self.get_velocity(x_t, t)

        # 1. Estymacja czystego obrazu
        x_estimate = ???

        # 2. Ocena artefaktów 
        loss = ???

        # Gradient odpychający od artefaktu
        grad_x = ???

        # Normalizacja
        grad_norm = ???

        # 3. Krok aktualizacji: x_{t-dt} = x_t - dt * (v + s * grad_norm)
        x_t_minus_dt = ???

        return x_t_minus_dt.detach()

# --- Test ---
sim_model = DummyFlowModel()
current_latent_x_t = torch.randn(1, 4, 64, 64)
t_current = 0.5
time_step_dt = 0.05
scale_of_correction = 5.0

next_latent_step = sim_model.step_diamond(
    x_t=current_latent_x_t,
    t=t_current,
    dt=time_step_dt,
    guidance_scale=scale_of_correction
)

assert next_latent_step.shape == (1, 4, 64, 64), \
    f"Zły kształt tensora: {next_latent_step.shape}"
assert not next_latent_step.requires_grad, \
    "Tensor nie powinien śledzić gradientów po .detach()"
assert not torch.isnan(next_latent_step).any(), \
    "Tensor zawiera NaN – sprawdź obliczenia gradientu"

print("Krok wykonany! Wymiary:", next_latent_step.shape)

Kolejna iteracja ukrytego 'czystego' po zredukowaniu krokem z korektą DIAMOND wykonana! Wymiary: torch.Size([1, 4, 64, 64])
